# Laboratorio integrador: analizador de tendencias en noticias argentinas

**Duración estimada:** 1 hora

## Desafío
Vas a construir un sistema en Python que extraiga noticias de la web y las procese con `spaCy` para identificar entidades, verbos frecuentes, palabras clave y visualizaciones básicas.


## Instalación de dependencias

In [1]:
# Instalamos las librerías necesarias en modo silencioso (-q)
!pip install spacy trafilatura pandas matplotlib wordcloud plotly -q
!python -m spacy download es_core_news_lg -q

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_lg')


## Importación de librerías

In [2]:
import spacy
import trafilatura
import pandas as pd
from collections import Counter
from datetime import datetime
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import plotly.graph_objects as go

nlp = spacy.load("es_core_news_lg")
print("Modelo y librerías cargadas.")

Modelo y librerías cargadas.


---
## Parte 1: extracción de noticias

**Objetivo:** construir una función que reciba una URL y devuelva un diccionario con el texto limpio de la noticia.

**Estrategia elegida:** `trafilatura` descarga el HTML y extrae el contenido principal automáticamente, filtrando menús, publicidades y boilerplate. Es más simple y robusta que usar `requests` + `BeautifulSoup` manualmente para este caso.

### Bitácora de IA — Parte 1
- **Objetivo:** elegir cómo descargar el texto de una URL periodística.
- **Prompt:** *"Dame dos estrategias para extraer el contenido principal de una página de noticias en Python."*
- **IA propuso:** (1) `requests` + `BeautifulSoup` con selectores CSS, (2) `trafilatura` con extracción automática.
- **Conservé:** la opción 2, porque evita depender de la estructura HTML de cada diario.
- **Descarté:** la opción 1 porque requiere inspeccionar el DOM de cada sitio por separado.
- **Aprendí:** `trafilatura` usa heurísticas para detectar el cuerpo principal del artículo.

In [3]:
def extraer_noticia(url):
    """
    Extrae el contenido principal de una noticia desde una URL.
    Devuelve un dict con 'url', 'texto' y 'fecha_extraccion', o None si falla.
    """
    try:
        # PASO 1: descargar el HTML de la página con trafilatura
        descargado = trafilatura.fetch_url(url)

        # PASO 2: extraer el texto principal a partir del HTML descargado
        texto = trafilatura.extract(descargado)

        if texto:
            return {
                'url': url,
                'texto': texto,
                'fecha_extraccion': datetime.now().strftime("%Y-%m-%d %H:%M")
            }

        print(f"Advertencia: no se pudo extraer texto de {url}")
        return None
    except Exception as e:
        print(f"Error procesando {url}: {e}")
        return None


# URLs de las noticias a analizar
urls_noticias = [
    "https://www.pagina12.com.ar/2025/11/16/cientificos-argentinos-abren-la-posibilidad-de-nuevos-tratamientos-contra-la-diabetes/",
    "https://www.clarin.com/estados-unidos/60-anos-uso-descubren-popular-farmaco-diabetes-actua-cerebro_0_ZCYAEIH7Vf.html",
]

# Prueba con la primera URL antes de procesar ambas
prueba = extraer_noticia(urls_noticias[0])
if prueba:
    print("Extracción exitosa. Primeros 300 caracteres:")
    print(prueba['texto'][:300])
else:
    print("No se pudo extraer la noticia. Verificá la URL o el acceso a internet.")

Extracción exitosa. Primeros 300 caracteres:
- Edición Impresa
- 50 Años del Golpe
- El País
- Economía
- Sociedad
- Deportes
- El Mundo
- Opinión
- Contratapa
- Recordatorios
- Cultura
- Cash
- Radio 750
- Buenos Aires|12
- Rosario|12
- Salta|12
- Argentina|12
- Radar
- Radar Libros
- Soy
- Las12
- No
- Negrx
- Ciencia
- Universidad
- Psicolo


---
## Parte 2: análisis de texto con spaCy

**Objetivo:** encapsular el análisis en una clase `AnalizadorNoticia`.

**Labels de entidades en `es_core_news_lg`:**
| Label | Significado |
|-------|-------------|
| `PER` | Persona |
| `ORG` | Organización |
| `LOC` | Lugar geográfico |
| `MISC` | Misceláneos (gentilicios, obras, etc.) |

### Bitácora de IA — Parte 2
- **Objetivo:** entender qué labels usa el modelo español y cómo filtrar verbos.
- **Prompt:** *"¿Qué valores tiene ent.label_ en el modelo es_core_news_lg de spaCy para personas, orgs y lugares?"*
- **IA propuso:** usar `PER`, `ORG`, `LOC` y `MISC`.
- **Conservé:** esos cuatro labels tal como los sugirió.
- **Corregí:** la IA mencionó también `GPE` (que es del modelo inglés); lo descarté porque no aplica al modelo español.
- **Aprendí:** los labels difieren entre modelos de idiomas distintos.

In [4]:
class AnalizadorNoticia:
    def __init__(self, texto, nlp_model):
        """
        Inicializa el analizador con un texto y un modelo de spaCy ya cargado.
        """
        self.texto_original = texto
        self.nlp = nlp_model

        # PASO 3: procesa el texto con el modelo de spaCy
        self.doc = self.nlp(texto)

    def obtener_entidades(self):
        """Devuelve un diccionario con entidades agrupadas por tipo."""
        entidades = {
            'PERSONAS': [],
            'ORGANIZACIONES': [],
            'LUGARES': [],
            'OTROS': []
        }

        # PASO 4: itera sobre las entidades reconocidas y clasifica cada una
        for ent in self.doc.ents:
            texto_ent = ent.text.strip()
            if ent.label_ == 'PER':
                entidades['PERSONAS'].append(texto_ent)
            elif ent.label_ == 'ORG':
                entidades['ORGANIZACIONES'].append(texto_ent)
            elif ent.label_ == 'LOC':
                entidades['LUGARES'].append(texto_ent)
            else:
                entidades['OTROS'].append(texto_ent)

        return entidades

    def obtener_verbos_principales(self, n=10):
        """Devuelve una lista de tuplas (verbo_lematizado, frecuencia)."""
        # PASO 5: filtra tokens que sean verbos, los lematiza y cuenta frecuencias
        verbos = [
            token.lemma_.lower()
            for token in self.doc
            if token.pos_ == 'VERB' and not token.is_stop and token.is_alpha
        ]
        frecuencias = Counter(verbos)
        return frecuencias.most_common(n)

    def obtener_estadisticas(self):
        """Calcula estadísticas descriptivas básicas del texto."""
        # PASO 6: calcula métricas del Doc de spaCy
        tokens_alpha = [t for t in self.doc if t.is_alpha]
        oraciones = list(self.doc.sents)
        palabras_unicas = set(t.lower_ for t in tokens_alpha)

        total_tokens = len(tokens_alpha)
        total_oraciones = len(oraciones)
        promedio = total_tokens / total_oraciones if total_oraciones > 0 else 0

        return {
            'total_tokens': total_tokens,
            'total_oraciones': total_oraciones,
            'palabras_unicas': len(palabras_unicas),
            'longitud_promedio_oracion': round(promedio, 2),
        }

    def extraer_frases_con_entidad(self, nombre_entidad):
        """Devuelve oraciones que contengan una entidad específica."""
        # PASO 7: recorre las oraciones y conserva las que mencionan la entidad
        oraciones = [
            sent.text.strip()
            for sent in self.doc.sents
            if nombre_entidad.lower() in sent.text.lower()
        ]
        return oraciones


# ─── Prueba con texto de ejemplo ────────────────────────────────────────────
texto_prueba = (
    "El presidente Javier Milei estuvo el martes en la provincia de Santa Fe para inaugurar "
    "una nueva planta de procesamiento de soja. La compañía AgroTech Argentina anunció una "
    "inversión de 80 millones de dólares. Mariana López, gerente general, explicó que el "
    "proyecto generará 300 puestos de trabajo."
)

analizador_prueba = AnalizadorNoticia(texto_prueba, nlp)
print("Entidades:", analizador_prueba.obtener_entidades())
print("Verbos:", analizador_prueba.obtener_verbos_principales())
print("Estadísticas:", analizador_prueba.obtener_estadisticas())

Entidades: {'PERSONAS': ['Javier Milei', 'Mariana López'], 'ORGANIZACIONES': ['AgroTech Argentina'], 'LUGARES': ['provincia de Santa Fe'], 'OTROS': ['La compañía']}
Verbos: [('inaugurar', 1), ('anunciar', 1), ('generar', 1)]
Estadísticas: {'total_tokens': 45, 'total_oraciones': 3, 'palabras_unicas': 36, 'longitud_promedio_oracion': 15.0}


---
## Parte 3: visualización de resultados

**Objetivo:** construir dos visualizaciones a partir del texto procesado.

### Bitácora de IA — Parte 3
- **Objetivo:** decidir qué tokens excluir de la nube de palabras.
- **Prompt:** *"¿Qué criterios uso para filtrar tokens en una nube de palabras con spaCy en español?"*
- **IA propuso:** excluir stopwords, puntuación, tokens no alfabéticos y tokens muy cortos (< 3 chars).
- **Conservé:** todos esos criterios; tienen sentido.
- **Agregué yo:** también excluir tokens que sean solo dígitos (`token.like_num`).
- **Aprendí:** usar `token.lemma_` en vez de `token.text` evita duplicados por conjugación.

In [5]:
def crear_nube_palabras(doc_procesado, titulo="Nube de palabras"):
    """Crea y muestra una nube de palabras a partir de un Doc de spaCy."""
    # PASO 8: extrae lemas relevantes filtrando ruido
    palabras_relevantes = [
        token.lemma_.lower()
        for token in doc_procesado
        if not token.is_stop
        and token.is_alpha
        and len(token.text) > 2
        and not token.like_num
    ]

    texto_limpio = ' '.join(palabras_relevantes)

    if texto_limpio:
        wordcloud = WordCloud(
            width=800,
            height=400,
            background_color='white',
            collocations=False,
        ).generate(texto_limpio)

        plt.figure(figsize=(10, 5))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis("off")
        plt.title(titulo, fontsize=14)
        plt.tight_layout()
        plt.show()
    else:
        print("No hay suficientes palabras relevantes para generar la nube.")


def visualizar_entidades_mas_comunes(entidades_dict, n=10, titulo="Entidades más frecuentes"):
    """Muestra un gráfico de barras horizontal con las entidades más frecuentes."""
    # PASO 9: combina todas las listas del diccionario y cuenta frecuencias
    todas_entidades = []
    for lista in entidades_dict.values():
        todas_entidades.extend(lista)

    frecuencias = Counter(todas_entidades)
    entidades_comunes = frecuencias.most_common(n)

    if not entidades_comunes:
        print("No se encontraron entidades para visualizar.")
        return

    # PASO 10: construir gráfico de barras horizontal con Plotly
    nombres = [item[0] for item in reversed(entidades_comunes)]
    conteos = [item[1] for item in reversed(entidades_comunes)]

    fig = go.Figure(
        go.Bar(
            x=conteos,
            y=nombres,
            orientation='h',
            marker_color='steelblue'
        )
    )
    fig.update_layout(
        title=titulo,
        xaxis_title="Frecuencia",
        yaxis_title="Entidad",
        height=400 + len(nombres) * 20,
        margin=dict(l=200)
    )
    fig.show()


print("Funciones de visualización definidas correctamente.")

Funciones de visualización definidas correctamente.


---
## Parte 4: integración en un pipeline

**Objetivo:** integrar extracción, análisis y reporte agregado para varias noticias.

### Bitácora de IA — Parte 4
- **Objetivo:** estructurar el pipeline con acumulación de entidades y verbos.
- **Prompt:** *"¿Cómo acumulo entidades de múltiples diccionarios con la misma estructura de llaves?"*
- **IA propuso:** iterar sobre `.items()` de cada dict y extender las listas.
- **Conservé:** esa lógica, es directa y legible.
- **Descarté:** una propuesta con `defaultdict` que era innecesariamente compleja para este caso.
- **Aprendí:** para acumular, `.extend()` es más claro que concatenar listas con `+`.

In [6]:
class AnalizadorTendencias:
    def __init__(self, lista_urls):
        self.urls = lista_urls
        self.noticias_data = []
        self.analizadores = []
        self.nlp = spacy.load("es_core_news_lg")

    def procesar_todas(self):
        """Orquesta la extracción y el análisis de todas las URLs."""
        print(f"Iniciando procesamiento de {len(self.urls)} URLs...")
        for url in self.urls:
            # PASO 11: extraer el texto de cada URL
            noticia = extraer_noticia(url)

            if noticia:
                self.noticias_data.append(noticia)

                # PASO 12: crear un AnalizadorNoticia y guardarlo
                analizador = AnalizadorNoticia(noticia['texto'], self.nlp)
                self.analizadores.append(analizador)
                print(f"  ✓ Procesada: {url[:70]}...")
            else:
                print(f"  ✗ No se pudo procesar: {url[:70]}...")

        print(f"\nProcesamiento completado: {len(self.analizadores)}/{len(self.urls)} noticias.")

    def generar_reporte_agregado(self, n=15):
        """Genera un reporte consolidado de todas las noticias procesadas."""
        if not self.analizadores:
            print("No hay noticias procesadas para generar un reporte.")
            return

        todas_las_entidades = {
            "PERSONAS": [],
            "ORGANIZACIONES": [],
            "LUGARES": [],
            "OTROS": []
        }
        todos_los_verbos = []

        # PASO 13: acumular entidades y verbos de todos los analizadores
        for analizador in self.analizadores:
            entidades = analizador.obtener_entidades()
            for tipo, lista in entidades.items():
                todas_las_entidades[tipo].extend(lista)

            verbos = analizador.obtener_verbos_principales(n=20)
            for verbo, freq in verbos:
                todos_los_verbos.extend([verbo] * freq)

        print(f"\n{'='*50}")
        print(f"REPORTE AGREGADO — {len(self.analizadores)} NOTICIAS")
        print(f"{'='*50}")

        # Estadísticas individuales
        for i, (noticia, analizador) in enumerate(zip(self.noticias_data, self.analizadores), 1):
            stats = analizador.obtener_estadisticas()
            print(f"\nNoticia {i}: {noticia['url'][:60]}...")
            print(f"  Tokens: {stats['total_tokens']} | Oraciones: {stats['total_oraciones']} | "
                  f"Palabras únicas: {stats['palabras_unicas']} | "
                  f"Long. promedio oración: {stats['longitud_promedio_oracion']}")

        print("\n--- ENTIDADES MÁS COMUNES ---")
        visualizar_entidades_mas_comunes(todas_las_entidades, n=n,
                                         titulo="Entidades más frecuentes — todas las noticias")

        print("\n--- VERBOS MÁS COMUNES ---")
        frecuencias_verbos = Counter(todos_los_verbos)
        print(frecuencias_verbos.most_common(n))

        print("\n--- NUBE DE PALABRAS COMBINADA ---")
        texto_combinado = " ".join(
            noticia['texto'] for noticia in self.noticias_data
        )
        doc_combinado = self.nlp(texto_combinado)
        crear_nube_palabras(doc_combinado, titulo="Nube de palabras — todas las noticias")


print("Clase AnalizadorTendencias definida correctamente.")

Clase AnalizadorTendencias definida correctamente.


---
## Ejecución del pipeline completo

In [ ]:
# URLs de las noticias sobre diabetes
urls_noticias = [
    "https://www.pagina12.com.ar/2025/11/16/cientificos-argentinos-abren-la-posibilidad-de-nuevos-tratamientos-contra-la-diabetes/",
    "https://www.clarin.com/estados-unidos/60-anos-uso-descubren-popular-farmaco-diabetes-actua-cerebro_0_ZCYAEIH7Vf.html",
]

pipeline = AnalizadorTendencias(urls_noticias)
pipeline.procesar_todas()
pipeline.generar_reporte_agregado()

Iniciando procesamiento de 2 URLs...
  ✓ Procesada: https://www.pagina12.com.ar/2025/11/16/cientificos-argentinos-abren-la...
  ✓ Procesada: https://www.clarin.com/estados-unidos/60-anos-uso-descubren-popular-fa...

Procesamiento completado: 2/2 noticias.

REPORTE AGREGADO — 2 NOTICIAS

Noticia 1: https://www.pagina12.com.ar/2025/11/16/cientificos-argentino...
  Tokens: 42 | Oraciones: 1 | Palabras únicas: 40 | Long. promedio oración: 42.0

Noticia 2: https://www.clarin.com/estados-unidos/60-anos-uso-descubren-...
  Tokens: 506 | Oraciones: 25 | Palabras únicas: 257 | Long. promedio oración: 20.24

--- ENTIDADES MÁS COMUNES ---


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

---
## Visualizaciones individuales por noticia

También podés ver el análisis de cada noticia por separado:

In [ ]:
# Análisis individual de cada noticia
for i, (noticia, analizador) in enumerate(zip(pipeline.noticias_data, pipeline.analizadores), 1):
    print(f"\n{'='*50}")
    print(f"Noticia {i}: {noticia['url'][:80]}")
    print(f"{'='*50}")

    entidades = analizador.obtener_entidades()
    print("\nPersonas mencionadas:", entidades['PERSONAS'][:5])
    print("Organizaciones:", entidades['ORGANIZACIONES'][:5])
    print("Lugares:", entidades['LUGARES'][:5])
    print("Verbos principales:", analizador.obtener_verbos_principales(5))

    # Nube de palabras individual
    crear_nube_palabras(analizador.doc, titulo=f"Noticia {i}")

    # Gráfico de entidades individual
    visualizar_entidades_mas_comunes(entidades, n=10, titulo=f"Entidades — Noticia {i}")

---
## Checklist de entrega

- [x] Funciones y clases completadas con código propio.
- [x] Prueba con noticias reales de Página 12 y Clarín.
- [x] Bitácora de interacción con IA completada en cada parte.
- [x] Al menos una decisión conservada y una descartada de las sugerencias de IA.
- [x] Código legible con docstrings y comentarios.